In [1]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                              Import Packages                                                                       #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

import boto3
import json
import pyodbc
import pandas as pd
from datetime import date, datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import plotly.express as px
import mplcursors
import plotly.graph_objects as go
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [3]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
SET DATEFIRST 1;  -- Monday = 1

WITH channels AS (
    SELECT DISTINCT
        CASE
            WHEN channel LIKE '%corr%' THEN 'Correspondent'
            ELSE channel
        END AS channel
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE channel NOT LIKE '%Credit Union Partners%'
),

base AS (
    SELECT
        CASE
            WHEN s.channel LIKE '%corr%' THEN 'Correspondent'
            ELSE s.channel
        END AS channel,
        s.funded_date,
        s.loan_amount,
        DATEPART(YEAR, s.funded_date) AS year_of_funding,
        DATEPART(WEEKDAY, s.funded_date) - 1 AS day_of_week,
        DATEPART(DAY, s.funded_date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, s.funded_date), 0), s.funded_date) + 1 AS week_of_month,
        DATEPART(MONTH, s.funded_date) AS month_of_year
    FROM marketing_sandbox.dbo.SDS s WITH (NOLOCK)
    WHERE s.funded_date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND s.funded_date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
      AND s.funded_date IS NOT NULL
      AND s.channel NOT LIKE '%Credit Union Partners%'
),

freq AS (
    SELECT
        channel,
        funded_date,
        loan_amount,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS loans_in_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS loans_in_month,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS loans_on_day_of_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS loans_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS amount_in_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS amount_in_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS amount_on_day_of_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS amount_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_loans_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_amount_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_loans_in_year,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_amount_in_year
    FROM base
),

agg_loans AS (
    SELECT
        channel,
        CAST(funded_date AS DATE) AS funded_date,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(*) AS number_of_loans,
        SUM(loan_amount) AS total_loan_amount,
        CAST(loans_in_week AS FLOAT) / NULLIF(loans_in_month, 0) AS week_weight,
        CAST(loans_on_day_of_week AS FLOAT) / NULLIF(loans_in_week, 0) AS day_of_week_weight,
        CAST(loans_on_day_of_month AS FLOAT) / NULLIF(loans_in_month, 0) AS day_of_month_weight,
        CAST(amount_in_week AS FLOAT) / NULLIF(amount_in_month, 0) AS week_amount_weight,
        CAST(amount_on_day_of_week AS FLOAT) / NULLIF(amount_in_week, 0) AS day_of_week_amount_weight,
        CAST(amount_on_day_of_month AS FLOAT) / NULLIF(amount_in_month, 0) AS day_of_month_amount_weight,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(SUM(total_loans_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(SUM(total_amount_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_amount,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(total_loans_in_year, 0) AS month_within_year_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(total_amount_in_year, 0) AS month_within_year_weight_amount
    FROM freq
    GROUP BY
        channel, CAST(funded_date AS DATE), year_of_funding, day_of_week,
        day_of_month, week_of_month, month_of_year, loans_in_week, loans_in_month,
        loans_on_day_of_week, loans_on_day_of_month, amount_in_week, amount_in_month,
        amount_on_day_of_week, amount_on_day_of_month, total_loans_in_month_across_years,
        total_amount_in_month_across_years, total_loans_in_year, total_amount_in_year
),

calendar_channels AS (
    SELECT
        c.Calendar_Date, ch.channel, c.Biz_Day, c.Funding_Day,
        c.Funding_Day_Remaining_in_Month, c.Biz_Day_Remaining_in_Month,
        c.Is_Holiday,
        DATEPART(WEEKDAY, c.Calendar_Date) - 1 AS day_of_week,
        DATEPART(DAY, c.Calendar_Date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, c.Calendar_Date), 0), c.Calendar_Date) + 1 AS week_of_month,
        DATEPART(MONTH, c.Calendar_Date) AS month_of_year
    FROM marketing_sandbox.dbo.Calendar c
    CROSS JOIN channels ch
    WHERE c.Calendar_Date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND c.Calendar_Date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
),

joined AS (
    SELECT
        cc.Calendar_Date, cc.channel, cc.Biz_Day, cc.Funding_Day,
        cc.Funding_Day_Remaining_in_Month, cc.Biz_Day_Remaining_in_Month,
        cc.Is_Holiday, cc.day_of_week, cc.day_of_month, cc.week_of_month,
        cc.month_of_year,
        ISNULL(al.number_of_loans, 0)            AS number_of_loans,
        ISNULL(al.total_loan_amount, 0)           AS total_loan_amount,
        ISNULL(al.week_weight, 0)                 AS week_weight,
        ISNULL(al.day_of_week_weight, 0)          AS day_of_week_weight,
        ISNULL(al.day_of_month_weight, 0)         AS day_of_month_weight,
        ISNULL(al.week_amount_weight, 0)          AS week_amount_weight,
        ISNULL(al.day_of_week_amount_weight, 0)   AS day_of_week_amount_weight,
        ISNULL(al.day_of_month_amount_weight, 0)  AS day_of_month_amount_weight,
        ISNULL(al.month_to_month_weight_loans, 0) AS month_to_month_weight_loans,
        ISNULL(al.month_to_month_weight_amount, 0)AS month_to_month_weight_amount,
        ISNULL(al.month_within_year_weight_loans, 0)  AS month_within_year_weight_loans,
        ISNULL(al.month_within_year_weight_amount, 0) AS month_within_year_weight_amount
    FROM calendar_channels cc
    LEFT JOIN agg_loans al
        ON cc.Calendar_Date = al.funded_date
       AND cc.channel = al.channel
)

SELECT
    Calendar_Date, channel, Biz_Day, Funding_Day,
    Funding_Day_Remaining_in_Month, Biz_Day_Remaining_in_Month,
    Is_Holiday, day_of_week, day_of_month, week_of_month, month_of_year,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE number_of_loans END          AS number_of_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE total_loan_amount END         AS total_loan_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_weight END               AS week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_weight END        AS day_of_week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_weight END       AS day_of_month_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_amount_weight END        AS week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_amount_weight END AS day_of_week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_amount_weight END AS day_of_month_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_loans END  AS month_to_month_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_amount END AS month_to_month_weight_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_loans END  AS month_within_year_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_amount END AS month_within_year_weight_amount
FROM joined
ORDER BY Calendar_Date ASC, channel;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (9150, 23)


In [4]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
WITH app_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS application_count,
        SUM(loan_amount) AS application_volume
    FROM (
        SELECT DISTINCT
            CAST(b.unified_app_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.unified_app_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

uw_event_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS underwriting_submission_events,
        SUM(loan_amount) AS underwriting_submission_event_volume
    FROM (
        SELECT DISTINCT
            CAST(b.underwriting_submission_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.underwriting_submission_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

approval_event_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS approval_events,
        SUM(loan_amount) AS approval_event_volume
    FROM (
        SELECT DISTINCT
            CAST(b.initial_conditional_approval_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.initial_conditional_approval_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

base AS (
    SELECT
        CAST(a.filedt AS DATE) AS filedt,
        a.channel,

        COUNT(DISTINCT a.loan_number) AS loan_count,
        SUM(a.loan_amount) AS loan_volume,

        COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END) AS funded_loan_count,
        SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END) AS funded_loan_volume,

        -- status-based counts
        COUNT(DISTINCT CASE WHEN b.underwriting_submission_date IS NOT NULL THEN a.loan_number END)
            AS underwriting_submission_count,

        COUNT(DISTINCT CASE WHEN b.initial_conditional_approval_date IS NOT NULL THEN a.loan_number END)
            AS initial_conditional_approval_count,

        -- event-based counts + volumes
        ISNULL(uw.underwriting_submission_events, 0) AS underwriting_submission_events,
        ISNULL(uw.underwriting_submission_event_volume, 0) AS underwriting_submission_event_volume,

        ISNULL(ap.approval_events, 0) AS approval_events,
        ISNULL(ap.approval_event_volume, 0) AS approval_event_volume,

        ISNULL(ac.application_count, 0) AS application_count,
        ISNULL(ac.application_volume, 0) AS application_volume,

        -- pull-through (count)
        CAST(
            100.0 * COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END)
            / NULLIF(COUNT(DISTINCT a.loan_number), 0)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_count,

        -- pull-through (volume)
        CAST(
            100.0 * SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END)
            / NULLIF(SUM(a.loan_amount), 0)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_volume,

        -- =========================
        -- AVERAGE LOAN SIZE METRICS
        -- =========================

        CAST(
            ISNULL(ac.application_volume, 0) * 1.0
            / NULLIF(ac.application_count, 0)
            AS DECIMAL(18,2)
        ) AS avg_application_loan_size,

        CAST(
            ISNULL(uw.underwriting_submission_event_volume, 0) * 1.0
            / NULLIF(uw.underwriting_submission_events, 0)
            AS DECIMAL(18,2)
        ) AS avg_underwriting_submission_loan_size,

        CAST(
            ISNULL(ap.approval_event_volume, 0) * 1.0
            / NULLIF(ap.approval_events, 0)
            AS DECIMAL(18,2)
        ) AS avg_approval_loan_size

    FROM marketing_sandbox.dbo.skinny_core a WITH (NOLOCK)

    JOIN marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        ON a.loan_number = b.loan_number

    LEFT JOIN app_counts ac
        ON CAST(a.filedt AS DATE) = ac.dt
        AND a.channel = ac.channel

    LEFT JOIN uw_event_counts uw
        ON CAST(a.filedt AS DATE) = uw.dt
        AND a.channel = uw.channel

    LEFT JOIN approval_event_counts ap
        ON CAST(a.filedt AS DATE) = ap.dt
        AND a.channel = ap.channel

    WHERE repeat_application = 0
      AND a.loan_status NOT LIKE '%C%'
      AND a.loan_status NOT IN (
            '0110','0115','0120','0125','0130','0140','0150','0160','0170','0180','0182',
            '0189','0190','0191','0192','0360','0362','0370','0375','0510','0610',
            '0618','0710','0711','0715','0720','1040'
      )

    GROUP BY 
        CAST(a.filedt AS DATE),
        a.channel,
        ac.application_count,
        ac.application_volume,
        uw.underwriting_submission_events,
        uw.underwriting_submission_event_volume,
        ap.approval_events,
        ap.approval_event_volume
)

SELECT
    b.*,

    c.Biz_Day,
    c.Biz_Day_Remaining_in_Month,
    c.Biz_Days_in_Month,
    c.Is_Holiday,
    c.Is_Company_Holiday,
    c.Calendar_Days_Remaining

FROM base b

LEFT JOIN marketing_sandbox.dbo.Calendar c
    ON b.filedt = CAST(c.Calendar_Date AS DATE)

WHERE c.Is_Weekday = 1

ORDER BY b.filedt;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        pipeline_df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {pipeline_df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (1623, 25)


In [5]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
SELECT * FROM marketing_sandbox.dbo.FDM_Anchor_Params WITH (NOLOCK);
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        anchor_df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {anchor_df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (89, 22)


In [6]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Data Preprocessing                                                        #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
df.fillna(0, inplace=True)

pipeline_df['filedt'] = pd.to_datetime(pipeline_df['filedt'])
pipeline_df.fillna(0, inplace=True)

In [7]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Split by Channel                                                          #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

wholesale_historical    = df[df['channel'] == 'Wholesale'].copy()
retail_historical       = df[df['channel'] == 'Retail'].copy()
retail_broker_historical= df[df['channel'] == 'Retail Broker'].copy()
corr_historical         = df[df['channel'] == 'Correspondent'].copy()

pipeline_wholesale    = pipeline_df[pipeline_df['channel'] == 'Wholesale'].copy()
pipeline_retail       = pipeline_df[pipeline_df['channel'] == 'Retail'].copy()
pipeline_retail_broker = pipeline_df[pipeline_df['channel'] == 'Retail Broker'].copy()

anchor_RB = anchor_df[anchor_df['channel'] == 'Retail Broker'].copy()
anchor_R = anchor_df[anchor_df['channel'] == 'Retail'].copy()
anchor_W = anchor_df[anchor_df['channel'] == 'Wholesale'].copy()

In [8]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
# Cast Decimal columns to float
#////////////////////////////////////////////////////////////////////////////////////////////////////////

# SQL DECIMAL/NUMERIC types arrive as Python decimal.Decimal — pandas can't do arithmetic on them
decimal_cols = pipeline_retail_broker.select_dtypes(include=['object']).columns.tolist()

# Force all numeric-looking columns to float
for col in pipeline_retail_broker.columns:
    if col not in ['filedt', 'channel', 'year_month']:
        pipeline_retail_broker[col] = pd.to_numeric(pipeline_retail_broker[col], errors='coerce').astype(float)

In [ ]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Set TODAY                                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

# For automation use:
# TODAY = pd.Timestamp(date.today())

# For backtesting, set manually:
TODAY = pd.Timestamp("2026-03-30")
TODAY = pd.to_datetime(TODAY)
print("Today's date is:", TODAY)

In [9]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          Feature schema definitions — Two-Window Profile 
#
#  CHANGES vs v10-A:
#  - FEATURE_COLS: restored the 6 _1m LEVEL features that v9 dropped.
#    Total dims now 25 (was 19 in v10-A / v9, 25 in v8).
#    This recovers v8's surge-detection (e.g. 9/30, 10/1) without
#    re-introducing v8's reversal blindness, because:
#       (a) _1m levels are HALF-WEIGHTED in the distance metric
#           via the new FEATURE_WEIGHTS dict (consumed by
#           RollingParamLookup._build_feature_weights).
#       (b) _1m pct_chg dims are still down-weighted on regime
#           reversals via the v9 turning-point dampener.
#
#  CHANGES vs v9 (already in v10-A):
#  - NORMALIZATION: median/MAD instead of mean/std.
#  - PER-FEATURE STD FLOOR.
#  - biz_day_count_forecast uses 2m window for both anchors AND live.
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

# ── Per-window FEATURE schema ─────────────────────────────────────────────────
_LEVEL_METRICS = [
    'apps_count_per_biz_day',
    'uw_count_per_biz_day',
    'appr_count_per_biz_day',
    'apps_vol_per_biz_day',
    'uw_vol_per_biz_day',
    'appr_vol_per_biz_day',
]

_PCT_CHG_METRICS = [
    'apps_count_pct_chg',
    'uw_count_pct_chg',
    'appr_count_pct_chg',
    'apps_vol_pct_chg',
    'uw_vol_pct_chg',
    'appr_vol_pct_chg',
]

_BASE_METRICS = _LEVEL_METRICS + _PCT_CHG_METRICS

# v10-B: restored _1m LEVEL features. 25 dims total.
FEATURE_COLS = (
    [f'{m}_1m' for m in _LEVEL_METRICS]   +   # 6 — short-term level (RESTORED, half-weight)
    [f'{m}_2m' for m in _LEVEL_METRICS]   +   # 6 — regime baseline
    [f'{m}_1m' for m in _PCT_CHG_METRICS] +   # 6 — short-term momentum (TPD-dampened on reversals)
    [f'{m}_2m' for m in _PCT_CHG_METRICS] +   # 6 — medium-term trajectory
    ['biz_day_count_forecast']                # 1 — calendar
)

# v10-B: static base weights applied inside RollingParamLookup distance calc.
# _1m levels get half-weight so they inform the metric without dominating it
# (which is what blew up v8 around the 10/30 reversal).
FEATURE_WEIGHTS = {
    **{f'{m}_1m': 0.50 for m in _LEVEL_METRICS},     # restored _1m levels: half-weight
    **{f'{m}_2m': 1.00 for m in _LEVEL_METRICS},     # regime baseline: full weight
    **{f'{m}_1m': 1.00 for m in _PCT_CHG_METRICS},   # momentum: full weight (TPD may multiply down)
    **{f'{m}_2m': 1.00 for m in _PCT_CHG_METRICS},   # trajectory: full weight
    'biz_day_count_forecast': 1.00,
}


# ── Single-window feature computation (UNCHANGED from v9/v10-A) ───────────────
def compute_volume_features(pipeline_df, window_start, window_end, prior_start, prior_end):
    pdf = pipeline_df.copy()
    pdf['filedt'] = pd.to_datetime(pdf['filedt'])

    curr  = pdf[(pdf['filedt'] >= pd.to_datetime(window_start)) &
                (pdf['filedt'] <= pd.to_datetime(window_end))]
    prior = pdf[(pdf['filedt'] >= pd.to_datetime(prior_start)) &
                (pdf['filedt'] <= pd.to_datetime(prior_end))]

    def safe_div(a, b):  return float(a / b) if b > 0 else 0.0
    def safe_pct(c, p):  return safe_div(c - p, p)

    biz_day_count = float(curr['Biz_Day'].sum()) if curr['Biz_Day'].sum() > 0 else float(len(curr))

    apps_count_total  = float(curr['application_count'].sum())
    uw_count_total    = float(curr['underwriting_submission_events'].sum())
    appr_count_total  = float(curr['approval_events'].sum())

    apps_count_per_bd = safe_div(apps_count_total, biz_day_count)
    uw_count_per_bd   = safe_div(uw_count_total,   biz_day_count)
    appr_count_per_bd = safe_div(appr_count_total, biz_day_count)

    apps_vol_total  = float(curr['application_volume'].sum())
    uw_vol_total    = float(curr['underwriting_submission_event_volume'].sum())
    appr_vol_total  = float(curr['approval_event_volume'].sum())

    apps_vol_per_bd = safe_div(apps_vol_total, biz_day_count)
    uw_vol_per_bd   = safe_div(uw_vol_total,   biz_day_count)
    appr_vol_per_bd = safe_div(appr_vol_total, biz_day_count)

    prior_biz_days = float(prior['Biz_Day'].sum()) if prior['Biz_Day'].sum() > 0 else float(len(prior))

    prior_apps_count = float(prior['application_count'].sum())
    prior_uw_count   = float(prior['underwriting_submission_events'].sum())
    prior_appr_count = float(prior['approval_events'].sum())

    prior_apps_cpbd = safe_div(prior_apps_count, prior_biz_days)
    prior_uw_cpbd   = safe_div(prior_uw_count,   prior_biz_days)
    prior_appr_cpbd = safe_div(prior_appr_count, prior_biz_days)

    prior_apps_vol = float(prior['application_volume'].sum())
    prior_uw_vol   = float(prior['underwriting_submission_event_volume'].sum())
    prior_appr_vol = float(prior['approval_event_volume'].sum())

    prior_apps_vpbd = safe_div(prior_apps_vol, prior_biz_days)
    prior_uw_vpbd   = safe_div(prior_uw_vol,   prior_biz_days)
    prior_appr_vpbd = safe_div(prior_appr_vol, prior_biz_days)

    return {
        'apps_count_per_biz_day': apps_count_per_bd,
        'apps_count_pct_chg':     safe_pct(apps_count_per_bd, prior_apps_cpbd),
        'uw_count_per_biz_day':   uw_count_per_bd,
        'uw_count_pct_chg':       safe_pct(uw_count_per_bd,   prior_uw_cpbd),
        'appr_count_per_biz_day': appr_count_per_bd,
        'appr_count_pct_chg':     safe_pct(appr_count_per_bd, prior_appr_cpbd),
        'apps_vol_per_biz_day':   apps_vol_per_bd,
        'apps_vol_pct_chg':       safe_pct(apps_vol_per_bd,   prior_apps_vpbd),
        'uw_vol_per_biz_day':     uw_vol_per_bd,
        'uw_vol_pct_chg':         safe_pct(uw_vol_per_bd,     prior_uw_vpbd),
        'appr_vol_per_biz_day':   appr_vol_per_bd,
        'appr_vol_pct_chg':       safe_pct(appr_vol_per_bd,   prior_appr_vpbd),
        'biz_day_count':          biz_day_count,
    }


# ── Two-window feature builders (UNCHANGED) ───────────────────────────────────
def compute_windowed_features(pipeline_df, run_date, lookback_days):
    end         = pd.to_datetime(run_date) - pd.Timedelta(days=1)
    start       = end - pd.Timedelta(days=lookback_days - 1)
    prior_end   = start - pd.Timedelta(days=1)
    prior_start = prior_end - pd.Timedelta(days=lookback_days - 1)
    return compute_volume_features(pipeline_df, start, end, prior_start, prior_end)


def compute_two_window_profile(pipeline_df, run_date, forecast_biz_days=None):
    """v10 fix A: biz_day_count_forecast uses 2m window for BOTH anchors and live."""
    f1 = compute_windowed_features(pipeline_df, run_date, lookback_days=30)
    f2 = compute_windowed_features(pipeline_df, run_date, lookback_days=60)

    out = {}
    for m in _BASE_METRICS:
        out[f'{m}_1m'] = f1[m]
        out[f'{m}_2m'] = f2[m]

    out['biz_day_count_forecast'] = f2['biz_day_count']
    return out

In [10]:
#///////////////////////////////////////////////////////////////////////////////
#  HISTORICAL CANDIDATE WINDOWS — full-history analog index
#
#  Replaces the FDM_Anchor_Params lookup. Instead of 30 pre-blessed anchors,
#  we generate a candidate window at every valid origin date and compute the
#  same two-window volume signature used by the live lookup.
#
#  No-lookahead guarantee:
#    - candidate_origin = the "as-if today" date for that historical point
#    - signature window = trailing 30/60d ending the day BEFORE candidate_origin
#    - forecast target  = 30 calendar days starting AT candidate_origin
#    - we only keep candidates whose forecast target window is fully inside
#      pipeline_retail_broker history (so we can score Optuna on it later)
#///////////////////////////////////////////////////////////////////////////////

def build_candidate_window_index(pipeline_df,
                                 history_df,
                                 step_days=1,
                                 min_history_days=90,
                                 forecast_horizon_days=30):
    """
    Returns a DataFrame with one row per candidate origin date, including
    its full two-window volume signature. Used as the analog search space.
    """
    pdf = pipeline_df.copy()
    pdf['filedt'] = pd.to_datetime(pdf['filedt'])

    hdf = history_df.copy()
    hdf['Calendar_Date'] = pd.to_datetime(hdf['Calendar_Date'])

    earliest = max(pdf['filedt'].min(), hdf['Calendar_Date'].min()) \
               + pd.Timedelta(days=min_history_days)
    latest_origin = hdf['Calendar_Date'].max() \
                    - pd.Timedelta(days=forecast_horizon_days)

    candidates = []
    for origin in pd.date_range(earliest, latest_origin, freq=f'{step_days}D'):
        # Skip weekends as origins to keep volume reasonable
        if origin.dayofweek in (5, 6):
            continue
        sig = compute_two_window_profile(pdf, run_date=origin)
        sig['candidate_origin'] = origin
        sig['target_start']     = origin
        sig['target_end']       = origin + pd.Timedelta(days=forecast_horizon_days - 1)
        candidates.append(sig)

    cand_df = pd.DataFrame(candidates)
    print(f"Built {len(cand_df)} historical candidate windows "
          f"({cand_df['candidate_origin'].min().date()} → "
          f"{cand_df['candidate_origin'].max().date()})")
    return cand_df


def fit_norm_stats_from_candidates(candidate_df):
    """
    Robust median/MAD normalization computed across the FULL candidate space
    rather than across 30 anchors. Same per-feature std-floor logic as v10-A.
    """
    norm_stats = {}
    for col in FEATURE_COLS:
        vals   = candidate_df[col].values.astype(float)
        median = float(np.median(vals))
        mad    = float(np.median(np.abs(vals - median)))
        robust_sd = 1.4826 * mad

        if col == 'biz_day_count_forecast':
            std_floor = 5.0
        elif col.endswith('_pct_chg_1m') or col.endswith('_pct_chg_2m'):
            std_floor = 0.05
        else:
            std_floor = max(abs(median) * 0.10, float(np.std(vals)) * 0.5)

        scale = max(robust_sd, std_floor, 1e-9)
        norm_stats[col] = {'mean': median, 'std': scale}
    return norm_stats


candidate_df = build_candidate_window_index(
    pipeline_df            = pipeline_retail_broker,
    history_df             = retail_broker_historical,
    step_days              = 1,           # daily granularity; bump to 5 if too slow
    min_history_days       = 90,          # need ≥60d for the 2m signature
    forecast_horizon_days  = 30,
)
norm_stats = fit_norm_stats_from_candidates(candidate_df)

Built 505 historical candidate windows (2024-05-01 → 2026-04-07)


In [11]:
#///////////////////////////////////////////////////////////////////////////////
#  AnalogMatcher — finds top-K historical windows most similar to live signature
#  Reuses FEATURE_COLS, FEATURE_WEIGHTS, turning-point dampener from v10-B.
#///////////////////////////////////////////////////////////////////////////////

class AnalogMatcher:

    TURNING_POINT_WEIGHT = 0.30

    def __init__(self, candidate_df, norm_stats):
        self.candidate_df = candidate_df.reset_index(drop=True).copy()
        self.norm_stats   = norm_stats
        self._base_weights = np.array(
            [FEATURE_WEIGHTS.get(c, 1.0) for c in FEATURE_COLS], dtype=float
        )
        self._pct_chg_1m_index = {
            m: FEATURE_COLS.index(f'{m}_1m') for m in _PCT_CHG_METRICS
        }
        # Pre-compute normalized candidate matrix once
        mat = []
        for _, row in self.candidate_df.iterrows():
            mat.append([(row[c] - norm_stats[c]['mean']) / norm_stats[c]['std']
                        for c in FEATURE_COLS])
        self._cand_matrix = np.array(mat)

    def _normalize(self, feats):
        return np.array([
            (feats.get(c, 0.0) - self.norm_stats[c]['mean']) / self.norm_stats[c]['std']
            for c in FEATURE_COLS
        ])

    def _build_weights(self, feats):
        w = self._base_weights.copy()
        reversed_metrics = []
        for m in _PCT_CHG_METRICS:
            v1, v2 = feats.get(f'{m}_1m', 0), feats.get(f'{m}_2m', 0)
            if abs(v1) > 0.02 and abs(v2) > 0.02 and v1 * v2 < 0:
                w[self._pct_chg_1m_index[m]] *= self.TURNING_POINT_WEIGHT
                reversed_metrics.append(m)
        return w, reversed_metrics

    def find_topk(self, today, pipeline_df, k=5, min_gap_days=14):
        """
        Returns top-k candidate windows by weighted distance, enforcing:
          - candidate_origin is BEFORE today (no lookahead)
          - candidates are spaced ≥ min_gap_days apart so we don't pick
            5 nearly-identical overlapping windows
        """
        today = pd.to_datetime(today)
        live_feats = compute_two_window_profile(pipeline_df, run_date=today)
        live_vec   = self._normalize(live_feats)
        weights, reversed_metrics = self._build_weights(live_feats)

        # Lookahead guard: candidate must end BEFORE today
        eligible_mask = self.candidate_df['target_end'] < today
        eligible_idx  = np.where(eligible_mask.values)[0]

        diffs = self._cand_matrix[eligible_idx] - live_vec
        dists = np.sqrt(((diffs ** 2) * weights).sum(axis=1))

        # Greedy top-k with min_gap spacing
        order = np.argsort(dists)
        picked, picked_origins = [], []
        for i in order:
            cand_row = self.candidate_df.iloc[eligible_idx[i]]
            origin   = cand_row['candidate_origin']
            if any(abs((origin - po).days) < min_gap_days for po in picked_origins):
                continue
            picked.append({
                'candidate_origin': origin,
                'target_start':     cand_row['target_start'],
                'target_end':       cand_row['target_end'],
                'distance':         float(dists[i]),
                'features':         {c: cand_row[c] for c in FEATURE_COLS},
            })
            picked_origins.append(origin)
            if len(picked) >= k:
                break

        print(f"\n[AnalogMatcher] today={today.date()}  live_signature found {len(picked)} matches")
        if reversed_metrics:
            print(f"  🔄 Turning-point dampener fired on: {reversed_metrics}")
        for rank, p in enumerate(picked, 1):
            print(f"  #{rank}  origin={p['candidate_origin'].date()}  "
                  f"target={p['target_start'].date()}→{p['target_end'].date()}  "
                  f"dist={p['distance']:.3f}")
        return picked, live_feats

In [12]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MASTER HOLIDAY CALENDAR (2021–2029)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def build_us_holiday_calendar(start_date='2021-01-01', end_date='2029-12-31'):
    """
    Builds full daily calendar spine with hard holiday flags (Is_Holiday).
    """
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    calendar   = pd.DataFrame({'Calendar_Date': full_dates})
    calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])

    cal      = USFederalHolidayCalendar()
    holidays = cal.holidays(start=start_date, end=end_date)

    calendar['Is_Holiday'] = calendar['Calendar_Date'].isin(holidays).astype(int)
    return calendar[['Calendar_Date', 'Is_Holiday']]

In [13]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          HolidayDistortionLearner
#  Learns per-(holiday_dow, business_day_offset) distortion factors from history.
#  e.g. "The Tuesday after a Monday holiday historically funds +35% above normal"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

class HolidayDistortionLearner:
    """
    Learns empirical holiday distortion factors from historical data.

    For each holiday in training history:
      - Identifies the holiday's day-of-week (holiday_dow)
      - Looks at surrounding business days in a window [-2, +4]
      - Computes ratio of actual / model-expected for each offset day
      - Aggregates across years using median + shrinkage toward 1.0

    Result: distortion_table keyed by (holiday_dow, biz_offset)
    """

    def __init__(self,
                 window_before=2,
                 window_after=4,
                 shrinkage=0.3,
                 min_observations=2):

        self.window_before = window_before
        self.window_after = window_after
        self.shrinkage = shrinkage
        self.min_observations = min_observations
        self.distortion_table_ = {}

    def fit(self, df, holiday_calendar, expected_col='expected'):
        """
        df            : training dataframe with Calendar_Date, actual target, and expected column
        holiday_calendar : dataframe with Calendar_Date and Is_Holiday columns
        expected_col  : column name in df containing the model's baseline expected value
        """

        df = df.copy()
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        holiday_calendar = holiday_calendar.copy()
        holiday_calendar['Calendar_Date'] = pd.to_datetime(
            holiday_calendar['Calendar_Date']
        )

        holiday_dates = holiday_calendar.loc[
            holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date'
        ].tolist()

        # Business day spine from training data (weekdays only, non-holiday)
        biz_days = df.loc[
            (~df['Calendar_Date'].dt.dayofweek.isin([5, 6]))
        ].copy()

        biz_days = biz_days.sort_values('Calendar_Date').reset_index(drop=True)

        # Map date -> position in business day sequence
        biz_day_index = {
            row['Calendar_Date']: idx
            for idx, row in biz_days.iterrows()
        }

        # Accumulate ratios: {(holiday_dow, offset): [ratio, ratio, ...]}
        ratio_accumulator = {}

        for hdate in holiday_dates:

            holiday_dow = hdate.dayofweek

            # Skip weekends (shouldn't happen with USFederalHolidayCalendar observed)
            if holiday_dow in [5, 6]:
                continue

            # Find surrounding business days
            for offset in range(-self.window_before, self.window_after + 1):

                if offset == 0:
                    continue  # Holiday itself is always hard zero

                # Step through calendar days to find the Nth business day offset
                candidate = hdate
                steps = 0
                direction = 1 if offset > 0 else -1
                abs_offset = abs(offset)

                while steps < abs_offset:
                    candidate = candidate + pd.Timedelta(days=direction)
                    if candidate.dayofweek not in [5, 6]:
                        steps += 1

                # Look up actual and expected for this candidate date
                row = df.loc[df['Calendar_Date'] == candidate]

                if row.empty:
                    continue

                actual = row[expected_col.replace('expected', df.columns[
                    [c for c in df.columns if 'loan_amount' in c or 'loans' in c][0]
                    if any('loan_amount' in c or 'loans' in c for c in df.columns)
                    else -1
                ])].values[0] if False else None

                # Simpler: use whatever target column is available
                target_cols = [c for c in df.columns
                               if 'total_loan_amount' in c or 'number_of_loans' in c]

                if not target_cols:
                    continue

                target_col = target_cols[0]
                actual = row[target_col].values[0]
                expected = row[expected_col].values[0]

                if expected <= 0 or actual < 0:
                    continue

                ratio = actual / expected

                key = (holiday_dow, offset)

                if key not in ratio_accumulator:
                    ratio_accumulator[key] = []

                ratio_accumulator[key].append(ratio)

        # Aggregate: median + shrinkage toward 1.0
        self.distortion_table_ = {}

        for key, ratios in ratio_accumulator.items():

            if len(ratios) < self.min_observations:
                continue

            median_ratio = np.median(ratios)

            shrunk = (
                (1 - self.shrinkage) * median_ratio +
                self.shrinkage * 1.0
            )

            self.distortion_table_[key] = shrunk

        return self

    def get_distortion(self, holiday_dow, biz_offset):
        """
        Returns learned distortion factor for a given
        (holiday day-of-week, business day offset from holiday).
        Defaults to 1.0 if no data learned.
        """
        return self.distortion_table_.get((holiday_dow, biz_offset), 1.0)

In [14]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#   Pipeline Data Preparation — NOW SUPPORTS MTD (TODAY - 1)
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def prepare_pipeline_monthly(pipeline_channel_df, today=None):

    pdf = pipeline_channel_df.copy()
    pdf['filedt'] = pd.to_datetime(pdf['filedt'])

    if today is not None:
        today = pd.to_datetime(today)
        cutoff = today - pd.tseries.offsets.BDay(1)
        pdf = pdf[pdf['filedt'] <= cutoff]

    pdf['year_month'] = pdf['filedt'].dt.to_period('M')

    # ── Force numeric ────────────────────────────────────────────────────
    for col in pdf.columns:
        if col not in ['filedt', 'channel', 'year_month']:
            pdf[col] = pd.to_numeric(pdf[col], errors='coerce')

    # ── Monthly sums (INCLUDING CURRENT PARTIAL MONTH) ───────────────────
    sum_cols = [
        'application_count',
        'approval_events',
        'underwriting_submission_events'
    ]

    monthly = pdf.groupby('year_month')[sum_cols].sum().reset_index()

    # ── Effective business days (CRITICAL FOR MTD) ───────────────────────
    pdf['is_biz_day'] = (
        (pdf['filedt'].dt.dayofweek < 5) &
        (pdf['Is_Holiday'] != 1) &
        (pdf['Is_Company_Holiday'] != 1)
    ).astype(int)

    biz_days = pdf.groupby('year_month')['is_biz_day'].sum().reset_index()
    biz_days = biz_days.rename(columns={'is_biz_day': 'effective_biz_days'})

    monthly = monthly.merge(biz_days, on='year_month', how='left')

    # Avoid divide-by-zero
    monthly['effective_biz_days'] = monthly['effective_biz_days'].clip(lower=1)

    # ── Per-biz-day normalization (NOW VALID FOR MTD) ────────────────────
    monthly['application_count_per_bizday'] = (
        monthly['application_count'] / monthly['effective_biz_days']
    )

    monthly['approval_events_per_bizday'] = (
        monthly['approval_events'] / monthly['effective_biz_days']
    )

    monthly['underwriting_submission_events_per_bizday'] = (
        monthly['underwriting_submission_events'] / monthly['effective_biz_days']
    )

    # ── Growth rates ─────────────────────────────────────────────────────
    monthly['application_count_per_bizday_growth_1m'] = (
        monthly['application_count_per_bizday'].pct_change()
    )

    monthly['application_count_per_bizday_growth_3m'] = (
        monthly['application_count_per_bizday'].pct_change(3)
    )

    monthly['approval_events_per_bizday_growth_1m'] = (
        monthly['approval_events_per_bizday'].pct_change()
    )

    monthly['underwriting_submission_events_per_bizday_growth_1m'] = (
        monthly['underwriting_submission_events_per_bizday'].pct_change()
    )

    monthly['year_month_str'] = monthly['year_month'].astype(str)

    return monthly

In [30]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#  StructuralLoanForecaster – Retail Broker v9
#  Changes vs v8:
#    - target_scale added: training data is divided by target_scale before
#      fitting so base_level operates in millions ($M) not raw dollars.
#      This makes the DOW-normalized scalar math work correctly since
#      implied_level and base_level are on the same scale.
#      Default 1_000_000 → base_level will be ~2-4 (meaning $2-4M/day avg).
#      Forecasts are multiplied back by target_scale before returning.
#    - holiday_shrinkage: 0.50 → 1.00 (fully disabled).
#      Distortion factors for RB are noisy and causing large errors on
#      Veterans Day, Columbus Day, Thanksgiving adjacents. With shrinkage=1.0
#      every factor collapses to 1.0 (no distortion applied).
#      Holiday hard-zero on the holiday itself is still preserved.
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

import numpy as np
import pandas as pd


class StructuralLoanForecaster:

    def __init__(self,
                 target='total_loan_amount',
                 target_scale=1_000_000,        # NEW: scale target to $M for numerical stability
                 dow_shrinkage=0.15,
                 interaction_shrinkage=0.60,
                 seasonal_exponent=1.02,
                 recency_strength=0.55,
                 trend_seasonal_tilt=0.12,
                 level_lookback_days=30,
                 trend_dampening=0.75,
                 forward_lift_daily=0.004,
                 holiday_window_before=1,
                 holiday_window_after=2,
                 holiday_shrinkage=1.00):       # 1.0 = fully disabled distortion factors

        self.target                = target
        self.target_scale          = target_scale
        self.dow_shrinkage         = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.seasonal_exponent     = seasonal_exponent
        self.recency_strength      = recency_strength
        self.trend_seasonal_tilt   = trend_seasonal_tilt
        self.level_lookback_days   = level_lookback_days
        self.trend_dampening       = trend_dampening
        self.forward_lift_daily    = forward_lift_daily
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after  = holiday_window_after
        self.holiday_shrinkage     = holiday_shrinkage

        self.master_calendar   = None
        self.distortion_table_ = {}
        self.fitted            = False

    # ---------------------------------------------------
    # INTERNAL: compute expected baseline for a dataframe
    # ---------------------------------------------------
    def _compute_expected(self, df, t_array):

        trend = np.exp(
            self.trend_intercept + self.trend_slope * t_array
        )

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        # Returns in $M (scaled units)
        return self.base_level * seasonal * trend * tilt

    # ---------------------------------------------------
    # FIT
    # ---------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date')
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Scale target to $M ----
        df[self.target] = df[self.target] / self.target_scale

        # ---- Store master holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar

            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        # ---- Production mask: weekdays, non-holiday, non-zero ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend (log-linear WLS, recency weighted) ----
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))

        y = np.log(df_prod[self.target].values)
        X = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W = np.diag(recency_prod)

        beta                 = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended']    = df[self.target] / trend_fitted

        # ---- Base level (exponentially weighted over lookback window) ----
        recent_prod = df.loc[prod_mask].tail(self.level_lookback_days).copy()
        n           = len(recent_prod)
        exp_weights = np.exp(np.linspace(0, 2.0, n))
        exp_weights /= exp_weights.sum()

        weighted_target = (recent_prod[self.target].values * exp_weights).sum()
        weighted_trend  = (recent_prod['trend_fitted'].values * exp_weights).sum()
        self.base_level = weighted_target / weighted_trend
        # base_level is now in $M, e.g. 2.5 means $2.5M average day

        # ---- DOW effect ----
        # Observed RB order: Mon > Wed ≈ Fri > Tue > Thu
        dow_raw = df.loc[prod_mask].groupby('dow')['detrended'].mean()

        dow_shrinkage_multipliers = {0: 0.50, 1: 1.60, 2: 0.80, 3: 0.90, 4: 0.80}

        dow_shrinkage_per_dow = {
            dow: self.dow_shrinkage * dow_shrinkage_multipliers.get(dow, 1.0)
            for dow in dow_raw.index
        }
        dow_shrunk = pd.Series({
            dow: (1 - dow_shrinkage_per_dow[dow]) * dow_raw[dow]
                 + dow_shrinkage_per_dow[dow] * self.base_level
            for dow in dow_raw.index
        })
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- MOY effect ----
        moy_raw         = df.loc[prod_mask].groupby('moy')['after_dow'].mean()
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dow'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction ----
        interaction_raw = (
            df.loc[prod_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = (
            interaction_shrunk / interaction_shrunk.mean()
        )

        # ================================================
        # LEARN HOLIDAY DISTORTION FACTORS
        # (holiday_shrinkage=1.0 collapses all to 1.0 — disabled)
        # ================================================
        if self.master_calendar is not None:

            t_full_arr     = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            holiday_dates = self.master_calendar.loc[
                self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
            ].tolist()

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            biz_day_pos = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in holiday_dates:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                if hdate not in biz_day_pos:
                    continue

                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    target_pos = h_pos + offset

                    if target_pos < 0 or target_pos >= len(biz_day_list):
                        continue

                    target_date = biz_day_list[target_pos]
                    row         = df.loc[df['Calendar_Date'] == target_date]

                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue

                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]

                    if expected <= 0 or actual <= 0:
                        continue

                    ratio = actual / expected
                    key   = (holiday_dow, offset)

                    if key not in ratio_accumulator:
                        ratio_accumulator[key] = []

                    ratio_accumulator[key].append(ratio)

            self.distortion_table_ = {}

            for key, ratios in ratio_accumulator.items():

                if len(ratios) < 2:
                    continue

                median_ratio = float(np.median(ratios))
                # holiday_shrinkage=1.0 → shrunk = 1.0 always (fully disabled)
                shrunk = (
                    (1 - self.holiday_shrinkage) * median_ratio +
                    self.holiday_shrinkage * 1.0
                )
                self.distortion_table_[key] = shrunk

        self.fitted = True
        return self

    # ---------------------------------------------------
    # FORECAST
    # ---------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit first.')

        start_date = pd.to_datetime(start_date)
        end_date   = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future = np.arange(self.n_train, self.n_train + len(df))

        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        # Forecast in $M
        forecast = (
            self.base_level
            * seasonal
            * trend_multiplier
            * tilt
            * forward_lift
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Apply holiday calendar ----
        if self.master_calendar is not None:

            df = df.merge(
                self.master_calendar,
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on holiday itself — still enforced
            forecast[df['Is_Holiday'].values == 1] = 0.0

            # Distortion factors all = 1.0 when holiday_shrinkage=1.0
            # so this loop is effectively a no-op but kept for future use
            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )

            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast

            distortion = np.ones(len(df))

            for hdate in all_relevant_holidays:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)

                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()

                    if not row_idx:
                        continue

                    idx    = row_idx[0]
                    factor = self.distortion_table_.get((holiday_dow, offset), 1.0)
                    distortion[idx] = factor

            forecast = forecast * distortion

        # ---- Convert back to dollars ----
        df['forecast'] = forecast * self.target_scale

        return df

In [31]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          PRODUCTION OVERRIDES — Retail Broker
#
#  Backtested 7 windows (2025-04 through 2025-10):
#    forward_lift_daily=0.000 + blend_k=2  →  Avg|err| 6.46%, Max|err| 13.5%
#    7/7 within ±15%, 5/7 within ±10%, 4/7 within ±5%
#////////////////////////////////////////////////////////////////////////////////////////////////////////

PHASE1_DEFAULTS = {
    'forward_lift_daily': 0.000,
}

PRODUCTION_BLEND_K = 2


def apply_phase1_overrides(params, override_config=None, verbose=False):
    if override_config is None:
        override_config = PHASE1_DEFAULTS
    new_params = dict(params)
    for key, new_val in override_config.items():
        if key in new_params:
            new_params[key] = new_val
    return new_params


def _add_business_days_local(start_date, n_days):
    current = pd.to_datetime(start_date)
    days_added = 0
    while days_added < n_days:
        current += pd.Timedelta(days=1)
        if current.dayofweek not in [5, 6]:
            days_added += 1
    return current

In [ ]:
#///////////////////////////////////////////////////////////////////////////////
#  Just-in-time Optuna — adapted from tune_model_a_15th_to_15th
#  Objective: minimize abs(% error) of the window TOTAL (matches your validated metric)
#///////////////////////////////////////////////////////////////////////////////

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def _warm_start_from_anchor(study, anchor_df_in, target_start):
    """Seed TPE with the nearest legacy anchor's params (informed first trial)."""
    if anchor_df_in is None or len(anchor_df_in) == 0:
        return
    a = anchor_df_in.copy()
    a['gap'] = (pd.to_datetime(a['window_start']) - pd.to_datetime(target_start)).abs()
    nearest = a.sort_values('gap').iloc[0]
    try:
        study.enqueue_trial({
            'recency_strength':       float(nearest['recency_strength']),
            'dow_shrinkage':          float(nearest['dow_shrinkage']),
            'interaction_shrinkage':  float(nearest['interaction_shrinkage']),
            'seasonal_exponent':      float(nearest['seasonal_exponent']),
            'trend_dampening':        float(nearest['trend_dampening']),
            'forward_lift_daily':     float(nearest['forward_lift_daily']),
            'level_lookback_days':    int(nearest['level_lookback_days']),
            'trend_seasonal_tilt':    float(nearest['trend_seasonal_tilt']),
        })
    except Exception as e:
        print(f"    (warm-start skipped: {e})")


def optimize_params_on_window(
    history_df,
    holiday_calendar,
    target_start,
    target_end,
    n_trials=200,
    timeout=None,
    seed=42,
    train_min_start='2021-09-01',
    target='total_loan_amount',
    warm_start_anchor_df=None,
    verbose=False,
    # Per-parameter bounds — same as your existing tuner
    recency_strength_range      = (0.0,  1.5),
    dow_shrinkage_range         = (0.0,  1.0),
    interaction_shrinkage_range = (0.0,  1.0),
    seasonal_exponent_range     = (0.7,  1.3),
    trend_dampening_range       = (0.0,  1.0),
    forward_lift_daily_range    = (0.0,  0.02),
    level_lookback_days_range   = (20,    90),
    trend_seasonal_tilt_range   = (0.0,  1.0),
):
    """
    Tune StructuralLoanForecaster params on an arbitrary [target_start, target_end]
    window using your existing 'abs % error of window total' objective.

    Returns: (best_params dict, best_abs_pct_err float, signed_pct_err float, actual_total, forecast_total)
    """
    target_start = pd.to_datetime(target_start)
    target_end   = pd.to_datetime(target_end)
    train_cutoff = target_start - pd.Timedelta(days=1)

    # ── Training slice (no lookahead) ─────────────────────────────────────────
    train_df = history_df[
        (history_df['Calendar_Date'] >= train_min_start) &
        (history_df['Calendar_Date'] <= train_cutoff)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy()
    train_df['Calendar_Date'] = pd.to_datetime(train_df['Calendar_Date'])

    if len(train_df) < 100:
        raise ValueError(f"Insufficient training data: {len(train_df)} rows "
                         f"(need ≥100) for target_start={target_start.date()}")

    # ── Actuals in the held-out window ────────────────────────────────────────
    actuals = history_df[
        (history_df['Calendar_Date'] >= target_start) &
        (history_df['Calendar_Date'] <= target_end) &
        (~history_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
        (history_df[target] > 0)
    ][['Calendar_Date', target]].copy()
    actuals['Calendar_Date'] = pd.to_datetime(actuals['Calendar_Date'])

    if actuals.empty:
        raise ValueError(f"No actuals in window {target_start.date()} → {target_end.date()}")

    actual_total = float(actuals[target].sum())
    months_forward = len(pd.period_range(target_start, target_end, freq='M')) + 1

    # ── Objective: abs % err of window total (matches your validated metric) ──
    def objective(trial):
        params = {
            'recency_strength':      trial.suggest_float('recency_strength',      *recency_strength_range),
            'dow_shrinkage':         trial.suggest_float('dow_shrinkage',         *dow_shrinkage_range),
            'interaction_shrinkage': trial.suggest_float('interaction_shrinkage', *interaction_shrinkage_range),
            'seasonal_exponent':     trial.suggest_float('seasonal_exponent',     *seasonal_exponent_range),
            'trend_dampening':       trial.suggest_float('trend_dampening',       *trend_dampening_range),
            'forward_lift_daily':    trial.suggest_float('forward_lift_daily',    *forward_lift_daily_range),
            'level_lookback_days':   trial.suggest_int('level_lookback_days',     *level_lookback_days_range),
            'trend_seasonal_tilt':   trial.suggest_float('trend_seasonal_tilt',   *trend_seasonal_tilt_range),
        }
        try:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                m = StructuralLoanForecaster(target=target, **params)
                m.fit(train_df, holiday_calendar)
                pred = m.forecast_from_date(start_date=target_start,
                                            months_forward=months_forward)
            pred = pred[(pred['Calendar_Date'] >= target_start) &
                        (pred['Calendar_Date'] <= target_end)]
            merged = actuals.merge(pred[['Calendar_Date', 'forecast']],
                                   on='Calendar_Date', how='inner')
            if merged.empty:
                return float('inf')
            forecast_total = merged['forecast'].sum()
            return abs(forecast_total - actual_total) / actual_total * 100
        except Exception:
            return float('inf')

    # ── Run study (with optional warm-start) ──────────────────────────────────
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    _warm_start_from_anchor(study, warm_start_anchor_df, target_start)
    study.optimize(objective, n_trials=n_trials, timeout=timeout,
                   show_progress_bar=False)

    best_params = study.best_trial.params
    best_abs_err = float(study.best_trial.value)

    # ── Re-run best params to capture signed err + forecast total ────────────
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        m = StructuralLoanForecaster(target=target, **best_params)
        m.fit(train_df, holiday_calendar)
        pred = m.forecast_from_date(start_date=target_start,
                                    months_forward=months_forward)
    pred = pred[(pred['Calendar_Date'] >= target_start) &
                (pred['Calendar_Date'] <= target_end)]
    merged = actuals.merge(pred[['Calendar_Date', 'forecast']],
                           on='Calendar_Date', how='inner')
    forecast_total = float(merged['forecast'].sum())
    signed_err = (forecast_total - actual_total) / actual_total * 100

    if verbose:
        print(f"    target  : {target_start.date()} → {target_end.date()}")
        print(f"    trials  : {len(study.trials)}  (warm-start={'yes' if warm_start_anchor_df is not None else 'no'})")
        print(f"    abs_err : {best_abs_err:.2f}%   signed: {signed_err:+.2f}%")
        print(f"    actual  : ${actual_total:,.0f}   forecast: ${forecast_total:,.0f}")
        print(f"    params  :")
        for k, v in best_params.items():
            if isinstance(v, float):
                print(f"      {k:<24s} = {v:.4f}")
            else:
                print(f"      {k:<24s} = {v}")

    return {
        'params':         best_params,
        'abs_pct_err':    best_abs_err,
        'signed_pct_err': signed_err,
        'actual_total':   actual_total,
        'forecast_total': forecast_total,
        'n_trials':       len(study.trials),
    }

In [33]:
#///////////////////////////////////////////////////////////////////////////////
#  ANALOG FORECAST — live driver
#  1. Match top-k historical windows to live volume signature
#  2. Run Optuna on each matched window (parallelizable)
#  3. Fit k models with the resulting params, forecast each
#  4. Ensemble forecasts weighted by inverse match distance
#///////////////////////////////////////////////////////////////////////////////

import io, contextlib


def produce_retail_broker_forecast_analog(today,
                                          retail_broker_historical,
                                          pipeline_retail_broker,
                                          candidate_df,
                                          norm_stats,
                                          holiday_calendar,
                                          k=3,
                                          n_trials_per_window=50,
                                          forecast_horizon_biz_days=20,
                                          train_start='2021-09-01',
                                          warm_start_anchor_df=None,  # legacy anchor_RB
                                          distance_warn_threshold=4.0,
                                          verbose=True):

    today          = pd.to_datetime(today)
    known_through  = _add_business_days_local(today, 3)
    forecast_start = _add_business_days_local(today, 4)
    forecast_end   = _add_business_days_local(forecast_start, forecast_horizon_biz_days - 1)

    if verbose:
        print(f"\n{'='*70}\n  ANALOG FORECAST — Retail Broker\n{'='*70}")
        print(f"  TODAY              : {today.date()}")
        print(f"  Forecast window    : {forecast_start.date()} → {forecast_end.date()}")
        print(f"  Top-k matches      : {k}")
        print(f"  Optuna trials/win  : {n_trials_per_window}")

    # ── 1. Match analogs ────────────────────────────────────────────────────
    matcher = AnalogMatcher(candidate_df, norm_stats)
    matches, live_feats = matcher.find_topk(forecast_start,
                                            pipeline_retail_broker,
                                            k=k)

    if matches[0]['distance'] > distance_warn_threshold:
        print(f"  🚨 Best match distance {matches[0]['distance']:.3f} > "
              f"{distance_warn_threshold}. Consider warm-start fallback.")

    # ── 2. Training data for the live forecast ──────────────────────────────
    train = retail_broker_historical[
        (retail_broker_historical['Calendar_Date'] >= train_start) &
        (retail_broker_historical['Calendar_Date'] <= known_through)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy()
    train['Calendar_Date'] = pd.to_datetime(train['Calendar_Date'])

    # ── 3. JIT-Optuna on each matched window, then forecast ─────────────────
    months_needed = len(pd.period_range(forecast_start, forecast_end, freq='M')) + 1
    per_match_forecasts = []

    for rank, m in enumerate(matches, 1):
        if verbose:
            print(f"\n  [Match #{rank}] Optimizing on "
                  f"{m['target_start'].date()}→{m['target_end'].date()} ...")

        # 1. Tune params on the matched historical window
        opt_result = optimize_params_on_window(
            history_df           = retail_broker_historical,
            holiday_calendar     = holiday_calendar,
            target_start         = m['target_start'],
            target_end           = m['target_end'],
            n_trials             = n_trials_per_window,
            warm_start_anchor_df = warm_start_anchor_df,
            verbose              = verbose,
        )
        applied_params = apply_phase1_overrides(opt_result['params'])

        # 2. Fit + forecast on LIVE training data using those tuned params
        model = StructuralLoanForecaster(**applied_params)
        ctx = (contextlib.redirect_stdout(io.StringIO())
               if not verbose else contextlib.nullcontext())
        with ctx:
            model.fit(train, holiday_calendar)
            pred = model.forecast_from_date(
                start_date     = forecast_start,
                months_forward = months_needed,
            )
        pred = pred[(pred['Calendar_Date'] >= forecast_start) &
                    (pred['Calendar_Date'] <= forecast_end)]

        # 3. Stash all diagnostics + forecast for ensembling
        per_match_forecasts.append({
            'rank':            rank,
            'match':           m,
            'params':          applied_params,
            'tune_abs_err':    opt_result['abs_pct_err'],
            'tune_signed_err': opt_result['signed_pct_err'],
            'tune_actual':     opt_result['actual_total'],
            'tune_forecast':   opt_result['forecast_total'],
            'n_trials':        opt_result['n_trials'],
            'forecast':        pred[['Calendar_Date', 'forecast']].copy(),
        })

    # ── 4. Inverse-distance ensemble ────────────────────────────────────────
    eps     = 1e-3
    raw_w   = np.array([1.0 / (p['match']['distance'] + eps) for p in per_match_forecasts])
    blend_w = raw_w / raw_w.sum()

    blended = per_match_forecasts[0]['forecast'][['Calendar_Date']].copy()
    blended['forecast'] = 0.0
    for p, w in zip(per_match_forecasts, blend_w):
        blended = blended.merge(
            p['forecast'].rename(columns={'forecast': 'f'}),
            on='Calendar_Date', how='left'
        )
        blended['forecast'] += blended['f'].fillna(0) * w
        blended = blended.drop(columns=['f'])

    if verbose:
        print(f"\n  Ensemble weights:")
        for p, w in zip(per_match_forecasts, blend_w):
            print(f"    #{p['rank']}  origin={p['match']['candidate_origin'].date()}  "
                  f"dist={p['match']['distance']:.3f}  weight={w:.3f}  "
                  f"tune_err={p['tune_abs_err']:.2f}%")

    # ── 5. Glue actuals + forecast for output (mirrors current schema) ──────
    actuals = retail_broker_historical[
        (retail_broker_historical['Calendar_Date'] >= today) &
        (retail_broker_historical['Calendar_Date'] <= known_through)
    ][['Calendar_Date', 'total_loan_amount']].copy()
    actuals['Calendar_Date'] = pd.to_datetime(actuals['Calendar_Date'])
    actuals = actuals.rename(columns={'total_loan_amount': 'amount'})
    actuals['source']     = 'actual'
    actuals['model_used'] = '—'

    forecast_out = blended.rename(columns={'forecast': 'amount'}).copy()
    forecast_out['source']     = 'forecast'
    forecast_out['model_used'] = f'RB_analog_k{k}'

    combined = pd.concat([actuals, forecast_out], ignore_index=True)
    combined = combined[~combined['Calendar_Date'].dt.dayofweek.isin([5, 6])]
    combined = combined.sort_values('Calendar_Date').reset_index(drop=True)
    combined['channel'] = 'Retail Broker'

    metadata = {
        'today':          today,
        'known_through':  known_through,
        'forecast_start': forecast_start,
        'forecast_end':   forecast_end,
        'matches':        per_match_forecasts,
        'blend_weights':  blend_w.tolist(),
        'live_features':  live_feats,
    }
    return combined, blended, metadata


# ── Build holiday calendar ────────────────────────────────────────────────────
holiday_calendar = build_us_holiday_calendar(start_date='2021-01-01', end_date='2030-12-31')

# ── Run it ────────────────────────────────────────────────────────────────────
combined_rb, forecast_rb, meta_rb = produce_retail_broker_forecast_analog(
    today                     = TODAY,
    retail_broker_historical  = retail_broker_historical,
    pipeline_retail_broker    = pipeline_retail_broker,
    candidate_df              = candidate_df,
    norm_stats                = norm_stats,
    holiday_calendar          = holiday_calendar,
    k                         = 3,
    n_trials_per_window       = 50,
    forecast_horizon_biz_days = 20,
    warm_start_anchor_df      = anchor_RB,   # keep old anchors as warm-start prior
    verbose                   = True,
)


  ANALOG FORECAST — Retail Broker
  TODAY              : 2026-03-30
  Forecast window    : 2026-04-03 → 2026-04-30
  Top-k matches      : 3
  Optuna trials/win  : 50

[AnalogMatcher] today=2026-04-03  live_signature found 3 matches
  #1  origin=2024-08-23  target=2024-08-23→2024-09-21  dist=1.957
  #2  origin=2024-08-09  target=2024-08-09→2024-09-07  dist=2.177
  #3  origin=2024-07-26  target=2024-07-26→2024-08-24  dist=2.653

  [Match #1] Optimizing on 2024-08-23→2024-09-21 ...
    target  : 2024-08-23 → 2024-09-21
    trials  : 50  (warm-start=yes)
    abs_err : 0.19%   signed: -0.19%
    actual  : $46,967,694   forecast: $46,878,570
    params  :
      recency_strength         = 4.5222
      dow_shrinkage            = 0.1041
      interaction_shrinkage    = 0.5635
      seasonal_exponent        = 0.0258
      trend_dampening          = 0.4523
      forward_lift_daily       = 0.3639
      level_lookback_days      = 17
      trend_seasonal_tilt      = 0.4161

  [Match #2] Optimizing 

In [34]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MAIN EXECUTION — Retail Broker (analog pipeline)
#  Consumes combined_rb / forecast_rb / meta_rb produced by Cell D.
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def add_business_days(start_date, n_days):
    current = pd.to_datetime(start_date)
    days_added = 0
    while days_added < n_days:
        current += pd.Timedelta(days=1)
        if current.dayofweek not in [5, 6]:
            days_added += 1
    return current

# Pull window boundaries straight from Cell D's metadata so they always agree
TODAY          = meta_rb['today']
KNOWN_THROUGH  = meta_rb['known_through']
FORECAST_START = meta_rb['forecast_start']
FORECAST_END   = meta_rb['forecast_end']
OUTPUT_THROUGH = add_business_days(TODAY, 29)

print(f"TODAY          : {TODAY.date()}")
print(f"Actuals through: {KNOWN_THROUGH.date()}")
print(f"Forecast start : {FORECAST_START.date()}")
print(f"Forecast end   : {FORECAST_END.date()}")
print(f"Output through : {OUTPUT_THROUGH.date()}")

# ── Pipeline monthly — diagnostic display only ────────────────────────────────
pipeline_monthly          = prepare_pipeline_monthly(pipeline_retail_broker, today=TODAY)
pipeline_monthly_complete = pipeline_monthly.copy()

print(f"\nPipeline months available (INCLUDING MTD): {len(pipeline_monthly_complete)}")
print(f"Latest month (may be partial): {pipeline_monthly_complete['year_month'].max()}")
print(f"\nLatest count-based indicators:")
latest = pipeline_monthly_complete.iloc[-1]
print(f"  application_count_per_bizday:                        {latest['application_count_per_bizday']:.1f}")
print(f"  approval_events_per_bizday_growth_1m:                {latest['approval_events_per_bizday_growth_1m']:+.3f}")
print(f"  underwriting_submission_events_per_bizday_growth_1m: {latest['underwriting_submission_events_per_bizday_growth_1m']:+.3f}")

# ── Analog match diagnostic ───────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  ANALOG MATCHES USED (from Cell D)")
print(f"{'='*70}")
print(f"  {'Rank':<5} {'Origin':<12} {'Target Window':<27} "
      f"{'Dist':>7} {'Weight':>7} {'TuneAbsErr':>11} {'TuneSignedErr':>14}")
print(f"  {'-'*86}")
for p, w in zip(meta_rb['matches'], meta_rb['blend_weights']):
    m = p['match']
    print(
        f"  #{p['rank']:<4} "
        f"{m['candidate_origin'].date()!s:<12} "
        f"{m['target_start'].date()!s} → {m['target_end'].date()!s}  "
        f"{m['distance']:>7.3f} "
        f"{w:>7.3f} "
        f"{p['tune_abs_err']:>10.2f}% "
        f"{p['tune_signed_err']:>+13.2f}%"
    )

print(f"\n{'='*70}")
print(f"  FORECAST OUTPUT")
print(f"{'='*70}")

# ── Trim combined_rb to the printable window (TODAY → OUTPUT_THROUGH) ────────
retail_broker_combined = combined_rb[
    (combined_rb['Calendar_Date'] >= TODAY) &
    (combined_rb['Calendar_Date'] <= OUTPUT_THROUGH) &
    (~combined_rb['Calendar_Date'].dt.dayofweek.isin([5, 6]))
].copy().sort_values('Calendar_Date').reset_index(drop=True)

# ── Print output ──────────────────────────────────────────────────────────────
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '         —'
    return f'{float(val):>12,.0f}'

header = (
    f"{'Date':<15} {'DOW':<6} {'Source':<10} {'Model':<30} "
    f"{'Channel':<14} {'Amount':>12}"
)
print(f"\n{header}")
print("-" * 95)

for _, row in retail_broker_combined.iterrows():
    dow_label = dow_names.get(row['Calendar_Date'].dayofweek, '—')
    print(
        f"{str(row['Calendar_Date'].date()):<15} "
        f"{dow_label:<6} "
        f"{row['source']:<10} "
        f"{str(row['model_used']):<30} "
        f"{row['channel']:<14} "
        f"{fmt(row['amount'])}"
    )

print("-" * 95)
actual_total   = retail_broker_combined.loc[
    retail_broker_combined['source'] == 'actual', 'amount'
].astype(float).sum()
forecast_total = retail_broker_combined.loc[
    retail_broker_combined['source'] == 'forecast', 'amount'
].astype(float).sum()
print(f"{'TOTAL actuals':<66} {fmt(actual_total)}")
print(f"{'TOTAL forecast':<66} {fmt(forecast_total)}")
print(f"{'TOTAL (actuals + forecast)':<66} {fmt(actual_total + forecast_total)}")

TODAY          : 2026-03-30
Actuals through: 2026-04-02
Forecast start : 2026-04-03
Forecast end   : 2026-04-30
Output through : 2026-05-08

Pipeline months available (INCLUDING MTD): 25
Latest month (may be partial): 2026-03

Latest count-based indicators:
  application_count_per_bizday:                        65.5
  approval_events_per_bizday_growth_1m:                +0.094
  underwriting_submission_events_per_bizday_growth_1m: +0.050

  ANALOG MATCHES USED (from Cell D)
  Rank  Origin       Target Window                  Dist  Weight  TuneAbsErr  TuneSignedErr
  --------------------------------------------------------------------------------------
  #1    2024-08-23   2024-08-23 → 2024-09-21    1.957   0.379       0.19%         -0.19%
  #2    2024-08-09   2024-08-09 → 2024-09-07    2.177   0.341       0.07%         +0.07%
  #3    2024-07-26   2024-07-26 → 2024-08-24    2.653   0.280       1.00%         +1.00%

  FORECAST OUTPUT

Date            DOW    Source     Model              

In [35]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                       BUILD EVAL_DF FROM CELL D OUTPUT (no re-fitting)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

VIEW_FROM = pd.to_datetime(TODAY)
VIEW_TO   = pd.to_datetime('2026-04-30')

print(f"Requested range: {VIEW_FROM.date()} → {VIEW_TO.date()}")

# Sanity check: did Cell D forecast far enough?
fc_max = forecast_rb['Calendar_Date'].max()
if fc_max < VIEW_TO:
    print(f"  ⚠️  forecast_rb only extends to {fc_max.date()}. "
          f"Bump `forecast_horizon_biz_days` in the Cell D call to cover {VIEW_TO.date()}.")

# Build a single label describing the analog ensemble used by Cell D
analog_label = (
    f"RB_analog_k{len(meta_rb['matches'])} ("
    + ", ".join(p['match']['candidate_origin'].strftime('%Y-%m-%d')
                for p in meta_rb['matches'])
    + ")"
)

# ── Pull actuals for the full requested window ────────────────────────────────
full_actuals = retail_broker_historical[
    (retail_broker_historical['Calendar_Date'] >= TODAY) &
    (retail_broker_historical['Calendar_Date'] <= VIEW_TO)
][['Calendar_Date', 'total_loan_amount']].copy()
full_actuals['Calendar_Date'] = pd.to_datetime(full_actuals['Calendar_Date'])
full_actuals = full_actuals.rename(columns={'total_loan_amount': 'actual'})

# ── Use forecast_rb directly (no re-fit) ──────────────────────────────────────
fc = forecast_rb.rename(columns={'forecast': 'forecast'}).copy()
fc['Calendar_Date'] = pd.to_datetime(fc['Calendar_Date'])
fc = fc[fc['Calendar_Date'] <= VIEW_TO].copy()
fc['model_used'] = analog_label

# ── Merge actuals onto forecast (outer-style: keep all forecast dates) ────────
eval_df = fc[['Calendar_Date', 'forecast', 'model_used']].merge(
    full_actuals, on='Calendar_Date', how='left'
)

eval_df['channel'] = 'Retail Broker'
eval_df['dow']     = eval_df['Calendar_Date'].dt.dayofweek

# Weekdays only, drop rows where forecast is zero/null
eval_df = eval_df[
    ~eval_df['Calendar_Date'].dt.dayofweek.isin([5, 6]) &
    (eval_df['forecast'].fillna(0) > 0)
].reset_index(drop=True)

# ── Source flag ───────────────────────────────────────────────────────────────
eval_df['source'] = eval_df['Calendar_Date'].apply(
    lambda d: 'actual' if d <= KNOWN_THROUGH else 'forecast'
)

# ── Known window: replace forecast with actuals ───────────────────────────────
known_mask = eval_df['Calendar_Date'] <= KNOWN_THROUGH
eval_df.loc[known_mask, 'forecast']   = eval_df.loc[known_mask, 'actual']
eval_df.loc[known_mask, 'model_used'] = 'actual'

# ── Error metrics ─────────────────────────────────────────────────────────────
eval_df['diff'] = eval_df.apply(
    lambda r: r['forecast'] - r['actual'] if pd.notna(r['actual']) else np.nan, axis=1
)
eval_df['pct_err'] = eval_df.apply(
    lambda r: (r['diff'] / r['actual']) * 100 if pd.notna(r['actual']) and r['actual'] > 0 else np.nan,
    axis=1
)

print(f"eval_df covers: {eval_df['Calendar_Date'].min().date()} → {eval_df['Calendar_Date'].max().date()}")
print(f"eval_df rows  : {len(eval_df)}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT TABLE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):,.0f}'

def fmt_pct(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):+.1f}%'

# ── Known actuals window ──────────────────────────────────────────────────────
known_actuals_print = retail_broker_historical[
    (retail_broker_historical['Calendar_Date'] >= TODAY) &
    (retail_broker_historical['Calendar_Date'] <= KNOWN_THROUGH) &
    (~retail_broker_historical['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (retail_broker_historical['total_loan_amount'] > 0)
][['Calendar_Date', 'total_loan_amount']].copy()

known_actuals_print['Calendar_Date'] = pd.to_datetime(known_actuals_print['Calendar_Date'])
known_actuals_print = known_actuals_print.rename(columns={'total_loan_amount': 'actual'})
known_actuals_print['source']     = 'actual'
known_actuals_print['channel']    = 'Retail Broker'
known_actuals_print['forecast']   = known_actuals_print['actual']
known_actuals_print['model_used'] = 'actual'
known_actuals_print['diff']       = 0.0
known_actuals_print['pct_err']    = 0.0

# ── Forecast rows ─────────────────────────────────────────────────────────────
forecast_rows = eval_df[eval_df['source'] == 'forecast'].copy()

# ── Combine ───────────────────────────────────────────────────────────────────
print_df = pd.concat(
    [known_actuals_print, forecast_rows], ignore_index=True
).sort_values(
    ['Calendar_Date', 'source'], ascending=[True, True]
).drop_duplicates(
    subset='Calendar_Date', keep='first'
).reset_index(drop=True)

# ── Apply date band filter ────────────────────────────────────────────────────
print_df_view = print_df[
    (print_df['Calendar_Date'] >= VIEW_FROM) &
    (print_df['Calendar_Date'] <= VIEW_TO)
].copy().reset_index(drop=True)

# ── Totals & MAE/MAPE — only rows where actuals exist ────────────────────────
has_actual_mask = (
    (print_df_view['source'] == 'forecast') &
    (print_df_view['actual'].notna()) &
    (print_df_view['actual'] > 0)
)
forecast_mask_view = print_df_view['source'] == 'forecast'

total_actual   = print_df_view.loc[has_actual_mask, 'actual'].sum()
total_forecast = print_df_view.loc[forecast_mask_view, 'forecast'].sum()
total_diff     = total_forecast - total_actual
total_pct      = (total_diff / total_actual) * 100 if total_actual > 0 else float('nan')

mae  = print_df_view.loc[has_actual_mask, 'diff'].abs().mean()    if has_actual_mask.any() else float('nan')
mape = print_df_view.loc[has_actual_mask, 'pct_err'].abs().mean() if has_actual_mask.any() else float('nan')

# ── Build table columns ───────────────────────────────────────────────────────
col_date, col_dow, col_source, col_model, col_channel = [], [], [], [], []
col_actual_out, col_forecast_out, col_diff_out, col_pct_out = [], [], [], []

for _, row in print_df_view.iterrows():
    col_date.append(str(row['Calendar_Date'].date()))
    col_dow.append(dow_names.get(row['Calendar_Date'].dayofweek, '—'))
    col_source.append(row['source'])
    col_model.append(str(row['model_used']))
    col_channel.append(row['channel'])
    col_actual_out.append(fmt(row.get('actual')))
    col_forecast_out.append(fmt(row['forecast']))
    col_diff_out.append(fmt(row.get('diff')))
    col_pct_out.append(fmt_pct(row.get('pct_err')))

# ── TOTAL row ─────────────────────────────────────────────────────────────────
col_date.append(f'TOTAL  ({VIEW_FROM.strftime("%b %d")} → {VIEW_TO.strftime("%b %d")})')
col_dow.append('')
col_source.append('forecast only')
col_model.append('')
col_channel.append('Retail Broker')
col_actual_out.append(fmt(total_actual))
col_forecast_out.append(fmt(total_forecast))
col_diff_out.append(fmt(total_diff))
col_pct_out.append(fmt_pct(total_pct))

# ── MAE row ───────────────────────────────────────────────────────────────────
col_date.append('MAE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual_out]:
    lst.append('')
col_forecast_out.append(fmt(mae))
col_diff_out.append('')
col_pct_out.append('')

# ── MAPE row ──────────────────────────────────────────────────────────────────
col_date.append('MAPE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual_out]:
    lst.append('')
col_forecast_out.append(fmt_pct(mape))
col_diff_out.append('')
col_pct_out.append('')

# ── Row fill colours ──────────────────────────────────────────────────────────
n_known_view    = print_df_view[print_df_view['source'] == 'actual'].shape[0]
n_forecast_view = print_df_view[print_df_view['source'] == 'forecast'].shape[0]
n_summary       = 3

fill_colors = (
    ['#f0f0f0'] * n_known_view    +
    ['white']   * n_forecast_view +
    ['#fffbea'] * n_summary
)

# ── Plotly table ──────────────────────────────────────────────────────────────
table_fig = go.Figure(data=[go.Table(
    columnwidth = [110, 50, 80, 220, 80, 100, 100, 100, 80],
    header = dict(
        values = [
            '<b>Date</b>', '<b>DOW</b>', '<b>Source</b>', '<b>Model</b>',
            '<b>Channel</b>', '<b>Actual</b>',
            '<b>Forecast</b>', '<b>Diff</b>', '<b>Pct</b>'
        ],
        fill_color = '#2a3f5f',
        font       = dict(color='white', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 30
    ),
    cells = dict(
        values = [
            col_date, col_dow, col_source, col_model, col_channel,
            col_actual_out, col_forecast_out, col_diff_out, col_pct_out
        ],
        fill_color = [fill_colors] * 9,
        font       = dict(color='black', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 26
    )
)])

table_fig.update_layout(
    title = dict(
        text = (
            f'Retail Broker — Actuals vs Analog Forecast | '
            f'{VIEW_FROM.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}'
        ),
        font = dict(size=13)
    ),
    width  = 1200,
    height = 80 + (n_known_view + n_forecast_view + n_summary) * 28,
    margin = dict(l=10, r=10, t=50, b=10)
)

table_fig.show()


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PLOT — ACTUALS vs FORECAST
#////////////////////////////////////////////////////////////////////////////////////////////////////////

fig = go.Figure()

# ── Actuals line ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x      = print_df['Calendar_Date'],
    y      = print_df['actual'],
    mode   = 'lines+markers',
    name   = 'Actual',
    line   = dict(color='black', width=2),
    marker = dict(size=6)
))

# ── Forecast line — analog ensemble is a single blended series ────────────────
forecast_only = print_df[print_df['source'] == 'forecast'].copy()
if not forecast_only.empty:
    fig.add_trace(go.Scatter(
        x      = forecast_only['Calendar_Date'],
        y      = forecast_only['forecast'],
        mode   = 'lines+markers',
        name   = f'Forecast — {analog_label}',
        line   = dict(color='steelblue', width=2.5, dash='dash'),
        marker = dict(size=6)
    ))

fig.add_vline(
    x=str(FORECAST_START.date()),
    line_width=1.5, line_dash='dash', line_color='grey'
)
fig.add_annotation(
    x=str(FORECAST_START.date()), y=1, yref='paper',
    text='Forecast Start', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)
fig.add_vrect(
    x0=str(TODAY.date()), x1=str(KNOWN_THROUGH.date()),
    fillcolor='lightgrey', opacity=0.2, layer='below', line_width=0
)
fig.add_annotation(
    x=str(TODAY.date()), y=1, yref='paper',
    text='Known Actuals', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)

fig.update_layout(
    title=dict(
        text=(
            f'Retail Broker — Actuals vs Forecast (Analog Matching + JIT Optuna)<br>'
            f'<sup>{TODAY.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}</sup>'
        ),
        font=dict(size=18)
    ),
    xaxis=dict(title='Date', tickformat='%b %d', tickangle=-45,
               showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title='Loan Volume ($)', tickformat='$,.0f',
               showgrid=True, gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    hovermode='x unified', plot_bgcolor='white', width=1200, height=550
)

fig.show()

Requested range: 2026-03-30 → 2026-04-30
eval_df covers: 2026-04-03 → 2026-04-30
eval_df rows  : 20


In [ ]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          BATCH BACKTEST — Multiple Date Ranges (Analog + JIT Optuna)
#
#  For each (range_start, range_end) window:
#    1. Treat range_start as TODAY
#    2. Use actuals through KNOWN_THROUGH (TODAY + 3 biz days)
#    3. Run analog matching + JIT Optuna against historical candidates that
#       end strictly before TODAY (lookahead-safe via AnalogMatcher)
#    4. Forecast the remainder of the window
#    5. Compare blended (actuals + forecast) total to true actuals total
#////////////////////////////////////////////////////////////////////////////////////////////////////////

date_ranges = [
    ('2025-10-01', '2025-10-31'),
    ('2025-10-10', '2025-11-10'),
    ('2025-10-20', '2025-11-20'),
    ('2025-10-30', '2025-11-30'),
    ('2025-11-01', '2025-11-30'),
    ('2025-11-10', '2025-12-10'),
    ('2025-11-20', '2025-12-20'),
    ('2025-11-30', '2025-12-30'),
    ('2025-12-01', '2025-12-31'),
    ('2025-12-10', '2026-01-10'),
    ('2025-12-20', '2026-01-20'),
    ('2025-12-30', '2026-01-30'),
    ('2026-01-01', '2026-01-31'),
    ('2026-01-10', '2026-02-10'),
    ('2026-01-20', '2026-02-20'),
    ('2026-01-30', '2026-02-28'),
    ('2026-02-01', '2026-02-28'),
    ('2026-02-10', '2026-03-10'),
    ('2026-02-20', '2026-03-20'),
    ('2026-03-01', '2026-03-31'),
    ('2026-03-10', '2026-04-10'),
    ('2026-03-20', '2026-04-20'),
    ('2026-03-30', '2026-04-30'),
]

# ── Backtest knobs (tune for speed vs. fidelity) ──────────────────────────────
K_BT          = 3     # number of analog matches per window
N_TRIALS_BT   = 50    # Optuna trials per matched window
MIN_GAP_BT    = 21    # min days between selected analog origins
USE_OVERRIDES = True  # apply PHASE1_DEFAULTS (forward_lift_daily=0) like prod

def _add_biz_days(d, n):
    cur, added = pd.to_datetime(d), 0
    while added < n:
        cur += pd.Timedelta(days=1)
        if cur.dayofweek not in [5, 6]:
            added += 1
    return cur

holiday_calendar_bt = build_us_holiday_calendar(
    start_date='2021-01-01', end_date='2030-12-31'
)
holiday_dates_set = set(pd.to_datetime(
    holiday_calendar_bt.loc[holiday_calendar_bt['Is_Holiday'] == 1, 'Calendar_Date']
))

# Single matcher reused across all windows (lookahead enforced inside .find_topk)
matcher_bt = AnalogMatcher(candidate_df, norm_stats)

results = []
import io, contextlib

for i, (start_str, end_str) in enumerate(date_ranges, 1):

    range_start = pd.to_datetime(start_str)
    range_end   = pd.to_datetime(end_str)

    bt_today          = range_start
    bt_known_through  = _add_biz_days(bt_today, 3)
    bt_forecast_start = _add_biz_days(bt_today, 4)
    actuals_end       = min(bt_known_through, range_end)

    print(f"\n[{i:>2}/{len(date_ranges)}] {range_start.date()} → {range_end.date()}  "
          f"(forecast from {bt_forecast_start.date()})")

    # ── Training data: history through KNOWN_THROUGH (no lookahead) ──────────
    train_bt = retail_broker_historical[
        (retail_broker_historical['Calendar_Date'] >= '2021-09-01') &
        (retail_broker_historical['Calendar_Date'] <= bt_known_through)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy()
    train_bt['Calendar_Date'] = pd.to_datetime(train_bt['Calendar_Date'])

    try:
        with contextlib.redirect_stdout(io.StringIO()):

            # ── 1. Find top-k analog matches (lookahead-safe) ────────────────
            matches_bt, _ = matcher_bt.find_topk(
                today        = bt_forecast_start,
                pipeline_df  = pipeline_retail_broker,
                k            = K_BT,
                min_gap_days = MIN_GAP_BT,
            )

            if not matches_bt:
                raise RuntimeError("no analog matches found")

            # ── 2. JIT-Optuna on each matched window, then forecast ──────────
            months_needed = len(pd.period_range(start=bt_forecast_start,
                                                end=range_end, freq='M')) + 1
            per_match = []
            for m in matches_bt:
                opt_result = optimize_params_on_window(
                    history_df           = retail_broker_historical,
                    holiday_calendar     = holiday_calendar_bt,
                    target_start         = m['target_start'],
                    target_end           = m['target_end'],
                    n_trials             = N_TRIALS_BT,
                    warm_start_anchor_df = anchor_RB,
                    verbose              = False,
                )
                applied = (apply_phase1_overrides(opt_result['params'])
                           if USE_OVERRIDES else opt_result['params'])
                model = StructuralLoanForecaster(**applied)
                model.fit(train_bt, holiday_calendar_bt)
                pred = model.forecast_from_date(start_date=bt_forecast_start,
                                                months_forward=months_needed)
                per_match.append({
                    'match':     m,
                    'tune_err':  opt_result['abs_pct_err'],
                    'forecast':  pred[['Calendar_Date', 'forecast']].copy(),
                })

            # ── 3. Inverse-distance ensemble ─────────────────────────────────
            eps   = 1e-3
            raw_w = np.array([1.0 / (p['match']['distance'] + eps) for p in per_match])
            w     = raw_w / raw_w.sum()

            blended = per_match[0]['forecast'][['Calendar_Date']].copy()
            blended['forecast'] = 0.0
            for p, wi in zip(per_match, w):
                blended = blended.merge(
                    p['forecast'].rename(columns={'forecast': 'f'}),
                    on='Calendar_Date', how='left'
                )
                blended['forecast'] += blended['f'].fillna(0) * wi
                blended = blended.drop(columns=['f'])

        # ── 4. Score: forecast portion of the range ──────────────────────────
        fc_mask = (
            (blended['Calendar_Date'] > actuals_end) &
            (blended['Calendar_Date'] <= range_end) &
            (~blended['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
            (~blended['Calendar_Date'].isin(holiday_dates_set))
        )
        forecast_total = float(blended.loc[fc_mask, 'forecast'].fillna(0).sum())

        # Build a compact analog label for the result row
        analog_label = " | ".join(
            f"{p['match']['candidate_origin'].strftime('%Y-%m-%d')}"
            f"(d={p['match']['distance']:.2f},err={p['tune_err']:.1f}%)"
            for p in per_match
        )
        mean_tune_err = float(np.mean([p['tune_err'] for p in per_match]))

        success = True
        err_msg = ''

    except Exception as e:
        forecast_total = float('nan')
        analog_label   = ''
        mean_tune_err  = float('nan')
        success        = False
        err_msg        = str(e)
        print(f"    ❌ failed: {err_msg}")

    # ── Actuals over the full and known portions ─────────────────────────────
    act_mask_full = (
        (retail_broker_historical['Calendar_Date'] >= range_start) &
        (retail_broker_historical['Calendar_Date'] <= range_end) &
        (~retail_broker_historical['Calendar_Date'].dt.dayofweek.isin([5, 6]))
    )
    actual_total = float(
        retail_broker_historical.loc[act_mask_full, 'total_loan_amount'].sum()
    )

    act_mask_known = (
        (retail_broker_historical['Calendar_Date'] >= range_start) &
        (retail_broker_historical['Calendar_Date'] <= actuals_end) &
        (~retail_broker_historical['Calendar_Date'].dt.dayofweek.isin([5, 6]))
    )
    actual_total_known = float(
        retail_broker_historical.loc[act_mask_known, 'total_loan_amount'].sum()
    )

    blended_total = actual_total_known + (forecast_total if success else 0.0)
    pct_diff = ((blended_total - actual_total) / actual_total * 100
                if actual_total > 0 and success else float('nan'))

    if success:
        print(f"    ✓ blended ${blended_total:>14,.0f}  vs actual ${actual_total:>14,.0f}  "
              f"({pct_diff:+.2f}%)  mean tune err {mean_tune_err:.2f}%")

    results.append({
        'date_range'     : f'{range_start.date()} → {range_end.date()}',
        'today'          : bt_today.date(),
        'actuals_through': actuals_end.date(),
        'forecast_from'  : (actuals_end + pd.Timedelta(days=1)).date(),
        'analogs_used'   : analog_label,
        'mean_tune_err%' : mean_tune_err,
        'actuals_total'  : actual_total,
        'forecast_total' : forecast_total,
        'blended_total'  : blended_total,
        'difference'     : blended_total - actual_total,
        'pct_difference' : pct_diff,
        'error'          : err_msg,
    })

backtest_df = pd.DataFrame(results)

# ── Pretty print ──────────────────────────────────────────────────────────────
disp = backtest_df.copy()
for col in ['actuals_total', 'forecast_total', 'blended_total', 'difference']:
    disp[col] = disp[col].apply(
        lambda v: f'{v:>15,.0f}' if pd.notna(v) else f'{"—":>15}'
    )
disp['pct_difference'] = disp['pct_difference'].apply(
    lambda v: f'{v:>14,.2f}%' if pd.notna(v) else f'{"—":>15}'
)
disp['mean_tune_err%'] = disp['mean_tune_err%'].apply(
    lambda v: f'{v:>6.2f}%' if pd.notna(v) else f'{"—":>7}'
)

# Drop the long analogs_used / error columns from the printed table for readability
print_cols = ['date_range', 'today', 'actuals_through', 'forecast_from',
              'mean_tune_err%', 'actuals_total', 'forecast_total',
              'blended_total', 'difference', 'pct_difference']
print(disp[print_cols].to_string(index=False))

# ── Aggregate accuracy summary ────────────────────────────────────────────────
ok = backtest_df.dropna(subset=['pct_difference']).copy()
if len(ok) > 0:
    abs_err = ok['pct_difference'].abs()
    print(f"\n{'─'*70}")
    print(f"  BACKTEST SUMMARY  ({len(ok)}/{len(backtest_df)} successful windows)")
    print(f"{'─'*70}")
    print(f"  Mean |pct_diff|  : {abs_err.mean():>6.2f}%")
    print(f"  Median |pct_diff|: {abs_err.median():>6.2f}%")
    print(f"  Max  |pct_diff|  : {abs_err.max():>6.2f}%")
    print(f"  Within ±5%       : {(abs_err <= 5).sum():>3}/{len(ok)}")
    print(f"  Within ±10%      : {(abs_err <= 10).sum():>3}/{len(ok)}")
    print(f"  Within ±15%      : {(abs_err <= 15).sum():>3}/{len(ok)}")
    print(f"  Mean tune err    : {ok['mean_tune_err%'].mean():>6.2f}%  "
          f"(Optuna abs err on matched historical windows)")

# ── Detailed analog table — uncomment to inspect which historicals were chosen
# print("\nAnalogs used per window:")
# for _, r in backtest_df.iterrows():
#     print(f"  {r['date_range']}: {r['analogs_used']}")


[ 1/23] 2025-10-01 → 2025-10-31  (forecast from 2025-10-07)
    ✓ blended $    54,947,912  vs actual $    76,662,287  (-28.32%)  mean tune err 2.18%

[ 2/23] 2025-10-10 → 2025-11-10  (forecast from 2025-10-16)
    ✓ blended $    67,060,207  vs actual $    77,104,956  (-13.03%)  mean tune err 2.58%

[ 3/23] 2025-10-20 → 2025-11-20  (forecast from 2025-10-24)
    ✓ blended $    81,682,721  vs actual $    81,447,909  (+0.29%)  mean tune err 2.31%

[ 4/23] 2025-10-30 → 2025-11-30  (forecast from 2025-11-05)
    ✓ blended $    32,842,513  vs actual $    67,217,542  (-51.14%)  mean tune err 0.35%

[ 5/23] 2025-11-01 → 2025-11-30  (forecast from 2025-11-06)
    ✓ blended $    34,821,465  vs actual $    60,127,797  (-42.09%)  mean tune err 0.47%

[ 6/23] 2025-11-10 → 2025-12-10  (forecast from 2025-11-14)
    ✓ blended $    50,271,920  vs actual $    64,306,238  (-21.82%)  mean tune err 0.67%

[ 7/23] 2025-11-20 → 2025-12-20  (forecast from 2025-11-26)
    ✓ blended $    37,012,109  vs actual